In [8]:
import numpy as np
import matplotlib
from matplotlib import pyplot as plot
from sklearn.linear_model import LinearRegression
from sklearn.decomposition import PCA
import sklearn.metrics
from sklearn.metrics import explained_variance_score, mean_squared_error, r2_score # https://scikit-learn.org/stable/auto_examples/linear_model/plot_ols.html
import statsmodels.api as sm
import pandas
import numpy as np
import matplotlib.pyplot as plt
import pandas
import scipy as sp
from scipy.odr import *
import random

# For 3d plots. This import is necessary to have 3D plotting below
from mpl_toolkits.mplot3d import Axes3D

# For statistics. Requires statsmodels 5.0 or more
from statsmodels.formula.api import ols
# Analysis of Variance (ANOVA) on linear models
from statsmodels.stats.anova import anova_lm

# Adapted from:
# https://www.datasciencecentral.com/profiles/blogs/linear-regression-in-python-use-of-numpy-scipy-and-statsmodels,
# https://realpython.com/linear-regression-in-python/#multiple-linear-regression


In [9]:
RawData = np.transpose(np.asarray(np.genfromtxt('Trabucchi_data_combined.csv',delimiter=',',skip_header=1)))
# 0 = logM
logMIndex = 0
logMData = RawData[logMIndex]
# 1 = logL
logLIndex = 1
logLData = RawData[logLIndex]
# 2 = logR
logRIndex = 2
logRData = RawData[logRIndex]
# 3 = logP0 from Trabucchi model
logP0Index = 3
logP0Data = RawData[logP0Index]
# 4 = logP1 from COLIBRI model
logP1Index=4
logP1Data = RawData[logP1Index]
# 5 = log(C/O)
logCORatioIndex=5
logCORatioData = RawData[logCORatioIndex]

In [10]:
## EXAMPLE

# Initiate some data, giving some randomness using random.random().
x = np.array([0, 1, 2, 3, 4, 5]) # create x values
y = np.array([i**2 + random.random() for i in x]) # create y values with y=x^2 + noise

# Define a function (quadratic in our case) to fit the data with.
def fitting_func(B, x):
    return B[0]*x[0] + B[1]*x[1] + B[2]

# Create a model for fitting.
linear_model = Model(fitting_func)

# Create a RealData object using our initiated data from above.
data = RealData(np.asarray([x,x**2]), y) # Input data is now x=(x, x^2), y = y

# Set up ODR with the model and data.
odr = ODR(data, linear_model, beta0=[0., 1.,2.]) # ODR function takes data, the model, and initial guesses for the coefficients as arguments

# Run the regression.
out = odr.run() #Do the fit

# Use the in-built pprint method to give us results.
out.pprint() # Prints everything in a readable form with pretty print
# In this case, we have
# 1. The Fitted Coefficients (as above in the fitting function?) AFTER TRANSFORM TO PRINCIPAL COMPONENTS
# 2. The errors on these coefficients
# 3. The covariance of the fitted coefficients (a nxn matrix)
# 4. The Residual variance (describes the quality of the fit. The lower, the better.)
# 5. The inverse condition #
# 6. Why did the model halt?

Beta: [0.35068158 0.95637501 0.13344416]
Beta Std Error: [0.1635283  0.03139144 0.17384106]
Beta Covariance: [[ 1.48177708 -0.27304006 -1.20156918]
 [-0.27304006  0.05460336  0.18206802]
 [-1.20156918  0.18206802  1.67456394]]
Residual Variance: 0.018046914686855372
Inverse Condition #: 0.0018491876436288475
Reason(s) for Halting:
  Sum of squares convergence


In [17]:
def PulsationModeSort (input_array):
    # DON'T TRUST THIS SORT METHOD. MODES IN DATA SET ARE UNRELIABLE
    FundamentalPulsators = []
    OvertonePulsators = []
    i = 0
    while i < len(input_array[ModeIndex]):
        if input_array[ModeIndex][i] == 0:
            FundamentalPulsators.append(i)
        if input_array[ModeIndex][i] == 1:
            OvertonePulsators.append(i)
        i += 1
    FundamentalModeStars = np.transpose(input_array.transpose()[FundamentalPulsators])
    FirstOvertoneModeStars = np.transpose(input_array.transpose()[OvertonePulsators])
    return FundamentalModeStars, FirstOvertoneModeStars

def CompositionSort (input_array):
    ORichStarsIndices = []
    CRichStarsIndices = []
    i = 0
    while i < len(input_array[logCORatioIndex]):
        if input_array[logCORatioIndex][i] < 0:
            ORichStarsIndices.append(i)
        if input_array[logCORatioIndex][i] > 0:
            CRichStarsIndices.append(i)
        i += 1
    ORichStars = np.transpose(input_array.transpose()[ORichStarsIndices])
    CRichStars = np.transpose(input_array.transpose()[CRichStarsIndices])
    return ORichStars, CRichStars

In [18]:
def MassSort (input_array):
    logMLT005 = []
    logMGT005 = []
    i = 0
    while i < len(input_array[logMIndex]):
        if input_array[logMIndex][i] <= 0.05: # For all stars with logM < 0.05
            logMLT005.append(i)
        if input_array[logMIndex][i] > -0.05 and input_array[logMIndex][i] <= 0.4 : # For all stars with logM > -0.05 - yes, there needs to be overlap!
            logMGT005.append(i)
        #if input_array[logMIndex][i] > 0.4:
            #print("Star", i, "of mass",10**(input_array[6][i]),"cut")
        i += 1
    #LowMassStars = np.transpose(input_array.transpose()[logMLT005])
    #HighMassStars = np.transpose(input_array.transpose()[logMGT005])
    return logMLT005, logMGT005

def MassFilter (Mass_Data):
    logMListArray = []
    i = 0
    while i<len(Mass_Data):
        if Mass_Data[i] < 0.4:
            logMListArray.append(i)
        i += 1
    return logMListArray

In [22]:
#P0Stars, P1Stars = PulsationModeSort(DataToFit)
ORichStars, CRichStars = CompositionSort(RawData)
DataToFit = ORichStars
LowLogMToFitStars, HighLogMToFitStars = MassSort(DataToFit)
logMData = DataToFit[logMIndex]
logMLT005Data = DataToFit[logMIndex][LowLogMToFitStars]
logMMGT005Data = DataToFit[logMIndex][HighLogMToFitStars]
logRData = DataToFit[logRIndex]
logRMLT005Data = DataToFit[logRIndex][LowLogMToFitStars]
logRMGT005Data = DataToFit[logRIndex][HighLogMToFitStars]
logLData = DataToFit[logLIndex]
logLLT005Data = DataToFit[logLIndex][LowLogMToFitStars]
logLMGT005Data = DataToFit[logLIndex][HighLogMToFitStars]
logP0Data = DataToFit[logP0Index]
logP1Data = DataToFit[logP1Index]

In [23]:
# First, a principal component analysis (PCA) for Period. To do this, we need to create an array made up of all the other (potentially) independent variables: M, L, R, and Log(C/O)
Period_PCA_Array = np.transpose(np.vstack((RawData[0],RawData[1],RawData[2],RawData[5])))
# Next, we can perform a PCA on the 4 components
pca = PCA(n_components=4)
pca.fit(Period_PCA_Array,RawData[3])
PCA(n_components=4)
print(pca.explained_variance_ratio_)
print(pca.singular_values_)

[0.67946943 0.27197297 0.04699722 0.00156037]
[23.24064425 14.70368116  6.1122243   1.11372377]


This seems to tell us that Mass accounts for about 68% of the observed variance and the luminosity accounts for 27%. The Radius and C/O ratio contribute minimally.

In [24]:
# Create a RealData object using our initiated data from above.
data = RealData(np.asarray([logMData,logRData]), logP0Data) # Data: x=[logM,logL], y=logP

# Set up ODR with the model and data.
odr = ODR(data, linear_model, beta0=[-0.78, 2.027,-2.23])

# Run the regression.
out = odr.run()

# Use the in-built pprint method to give us results.
out.pprint()

Beta: [-0.78149607  2.02878315 -2.23916189]
Beta Std Error: [0.00211792 0.00217209 0.00469197]
Beta Covariance: [[ 0.25245722 -0.16206188  0.32384173]
 [-0.16206188  0.26553595 -0.57147015]
 [ 0.32384173 -0.57147015  1.23901897]]
Residual Variance: 1.776774488346172e-05
Inverse Condition #: 0.01949132701523857
Reason(s) for Halting:
  Sum of squares convergence


ODR: logP0 = -0.781(2) logM + 2.028(2) logR - 2.223(4);
MLR: logP = -2.234(5) - 0.780(2)logM + 2.027(2)logR

In [25]:
# Next, a PCA for radius. To do this, we need to create an array made up of all the other (potentially) independent variables: M, L, P0, and Log(C/O)
Radius_PCA_Array = np.transpose(np.vstack((RawData[0],RawData[1],RawData[3],RawData[5])))
# Next, we can perform a PCA on the 5 components
pca = PCA(n_components=4)
pca.fit(Radius_PCA_Array,RawData[2])
PCA(n_components=4)
print(pca.explained_variance_ratio_)
print(pca.singular_values_)

[0.61096418 0.3051     0.08224994 0.00168588]
[24.49409274 17.30910409  8.98713532  1.28666751]


This seems to suggest we are correct in assuming that the mass and luminosity are the main contributors to the variation in radius (i.e., we have an RLM relation)

In [27]:
# Create a RealData object using our initiated data from above.
data = RealData(np.asarray([logMData,logLData]), logRData) # Data: x=[logM,logL], y=logR

# Set up ODR with the model and data.
odr = ODR(data, linear_model, beta0=[0., 1.,2.])

# Run the regression.
out = odr.run()

# Use the in-built pprint method to give us results.
out.pprint()

Beta: [-0.25904507  0.69506042 -0.25804086]
Beta Std Error: [0.00433791 0.00261414 0.00909208]
Beta Covariance: [[ 0.09597284 -0.04354314  0.14424779]
 [-0.04354314  0.03485348 -0.12086563]
 [ 0.14424779 -0.12086563  0.42161373]]
Residual Variance: 0.00019607046570124163
Inverse Condition #: 0.0015970253105102932
Reason(s) for Halting:
  Sum of squares convergence


ODR: logR = -0.258(9) + 0.695(2)logL - 0.259(4)logM

MLR: logR = -0.241(9) + 0.690(3)logL - 0.251(4)logM


In [39]:
X = np.array([[-1, -1,-1], [-2, -1,-1], [-3, -2,-2], [1, 1,1], [2, 1,1], [3, 2,3],[1,1,1]])
pca = PCA(n_components=3)
pca.fit(X)
PCA(n_components=3)
print(pca.explained_variance_ratio_)
print(pca.singular_values_)

[0.98423449 0.01219992 0.00356559]
[7.62957666 0.84943399 0.45921572]


In [40]:
pca = PCA(n_components=3, svd_solver='full')
pca.fit(X)
PCA(n_components=3, svd_solver='full')
print(pca.explained_variance_ratio_)
print(pca.singular_values_)

[0.98423449 0.01219992 0.00356559]
[7.62957666 0.84943399 0.45921572]


In [41]:
pca = PCA(n_components=1, svd_solver='arpack')
pca.fit(X)
PCA(n_components=1, svd_solver='arpack')
print(pca.explained_variance_ratio_)
print(pca.singular_values_)

[0.98423449]
[7.62957666]


In [42]:
def PulsationModeSort (input_array):
    # DON'T TRUST THIS SORT METHOD. MODES IN DATA SET ARE UNRELIABLE
    FundamentalPulsators = []
    OvertonePulsators = []
    i = 0
    while i < len(input_array[ModeIndex]):
        if input_array[ModeIndex][i] == 0:
            FundamentalPulsators.append(i)
        if input_array[ModeIndex][i] == 1:
            OvertonePulsators.append(i)
        i += 1
    FundamentalModeStars = np.transpose(input_array.transpose()[FundamentalPulsators])
    FirstOvertoneModeStars = np.transpose(input_array.transpose()[OvertonePulsators])
    return FundamentalModeStars, FirstOvertoneModeStars

def CompositionSort (input_array):
    ORichStarsIndices = []
    CRichStarsIndices = []
    i = 0
    while i < len(input_array[logCORatioIndex]):
        if input_array[logCORatioIndex][i] < 0:
            ORichStarsIndices.append(i)
        if input_array[logCORatioIndex][i] > 0:
            CRichStarsIndices.append(i)
        i += 1
    ORichStars = np.transpose(input_array.transpose()[ORichStarsIndices])
    CRichStars = np.transpose(input_array.transpose()[CRichStarsIndices])
    return ORichStars, CRichStars

In [43]:
def MassSort (input_array):
    logMLT005 = []
    logMGT005 = []
    i = 0
    while i < len(input_array[logMIndex]):
        if input_array[logMIndex][i] <= 0.05: # For all stars with logM < 0.05
            logMLT005.append(i)
        if input_array[logMIndex][i] > -0.05 and input_array[logMIndex][i] <= 0.4 : # For all stars with logM > -0.05 - yes, there needs to be overlap!
            logMGT005.append(i)
        #if input_array[logMIndex][i] > 0.4:
            #print("Star", i, "of mass",10**(input_array[6][i]),"cut")
        i += 1
    #LowMassStars = np.transpose(input_array.transpose()[logMLT005])
    #HighMassStars = np.transpose(input_array.transpose()[logMGT005])
    return logMLT005, logMGT005

def MassFilter (Mass_Data):
    logMListArray = []
    i = 0
    while i<len(Mass_Data):
        if Mass_Data[i] < 0.4:
            logMListArray.append(i)
        i += 1
    return logMListArray

In [65]:
#P0Stars, P1Stars = PulsationModeSort(DataToFit)
ORichStars, CRichStars = CompositionSort(RawData)
DataToFit = ORichStars
LowLogMToFitStars, HighLogMToFitStars = MassSort(DataToFit)
logMData = DataToFit[logMIndex]
logMLT005Data = DataToFit[logMIndex][LowLogMToFitStars]
logMMGT005Data = DataToFit[logMIndex][HighLogMToFitStars]
logRData = DataToFit[logRIndex]
logRMLT005Data = DataToFit[logRIndex][LowLogMToFitStars]
logRMGT005Data = DataToFit[logRIndex][HighLogMToFitStars]
logLData = DataToFit[logLIndex]
logLLT005Data = DataToFit[logLIndex][LowLogMToFitStars]
logLMGT005Data = DataToFit[logLIndex][HighLogMToFitStars]
logP0Data = DataToFit[logP0Index]
logP1Data = DataToFit[logP1Index]
logCORatioData = DataToFit[logCORatioIndex]

In [67]:
def fitting_func(B, x):
    return B[0]*x[0] + B[1]*x[1] + B[2]

# Create a RealData object using our initiated data from above.
data = RealData(np.asarray([logMData,logLData,logCORatioData]), logP0Data) # Data: x=[logM,logL], y=logP

# Set up ODR with the model and data.
odr = ODR(data, linear_model, beta0=[0., 0.,0.])

# Run the regression.
out = odr.run()

# Use the in-built pprint method to give us results.
out.pprint()

Beta: [-1.32535261  1.41942761 -2.7938375 ]
Beta Std Error: [0.0087485  0.00526641 0.01831129]
Beta Covariance: [[ 0.29885042 -0.13591669  0.45037627]
 [-0.13591669  0.10829699 -0.37545115]
 [ 0.45037627 -0.37545115  1.30925765]]
Residual Variance: 0.0002561019948123722
Inverse Condition #: 0.011074087865783991
Reason(s) for Halting:
  Sum of squares convergence


In [48]:
# Create a RealData object using our initiated data from above.
data = RealData(np.asarray([logMData,logLData]), logRData) # Data: x=[logM,logL], y=logR

# Set up ODR with the model and data.
odr = ODR(data, linear_model, beta0=[0., 1.,2.])

# Run the regression.
out = odr.run()

# Use the in-built pprint method to give us results.
out.pprint()

Beta: [-0.25904507  0.69506042 -0.25804086]
Beta Std Error: [0.00433791 0.00261414 0.00909208]
Beta Covariance: [[ 0.09597284 -0.04354314  0.14424779]
 [-0.04354314  0.03485348 -0.12086563]
 [ 0.14424779 -0.12086563  0.42161373]]
Residual Variance: 0.00019607046570124163
Inverse Condition #: 0.0015970253105102932
Reason(s) for Halting:
  Sum of squares convergence


In [28]:
#P0Stars, P1Stars = PulsationModeSort(DataToFit)
DataToFit = CRichStars
LowLogMToFitStars, HighLogMToFitStars = MassSort(DataToFit)
logMData = DataToFit[logMIndex]
logMLT005Data = DataToFit[logMIndex][LowLogMToFitStars]
logMMGT005Data = DataToFit[logMIndex][HighLogMToFitStars]
logRData = DataToFit[logRIndex]
logRMLT005Data = DataToFit[logRIndex][LowLogMToFitStars]
logRMGT005Data = DataToFit[logRIndex][HighLogMToFitStars]
logLData = DataToFit[logLIndex]
logLLT005Data = DataToFit[logLIndex][LowLogMToFitStars]
logLMGT005Data = DataToFit[logLIndex][HighLogMToFitStars]
logP0Data = DataToFit[logP0Index]
logP1Data = DataToFit[logP1Index]

In [30]:
# Create a RealData object using our initiated data from above.
data = RealData(np.asarray([logMData,logRData]), logP0Data) # Data: x=[logM,logR], y=logP

# Set up ODR with the model and data.
odr = ODR(data, linear_model, beta0=[0., 0.,0.])

# Run the regression.
out = odr.run()

# Use the in-built pprint method to give us results.
out.pprint()

Beta: [-0.82819699  2.04629214 -2.29246188]
Beta Std Error: [0.00430689 0.00299673 0.00721649]
Beta Covariance: [[ 0.42961388  0.03956618 -0.16319674]
 [ 0.03956618  0.20799233 -0.49740875]
 [-0.16319674 -0.49740875  1.20615608]]
Residual Variance: 4.317661854149892e-05
Inverse Condition #: 0.022417500821603785
Reason(s) for Halting:
  Sum of squares convergence


ODR: logP0 = -0.828(4) logM + 2.046(2) logR - 2.292(7)
MLR: logP0 = -0.826(4) logM + 2.043(3) logR - 2.285(7) 

In [32]:
# Create a RealData object using our initiated data from above.
data = RealData(np.asarray([logMData,logLData]), logRData) # Data: x=[logM,logL], y=logR

# Set up ODR with the model and data.
odr = ODR(data, linear_model, beta0=[0., 1.,2.])

# Run the regression.
out = odr.run()

# Use the in-built pprint method to give us results.
out.pprint()

Beta: [-0.42558829  0.76544232 -0.42111721]
Beta Std Error: [0.01383075 0.00744828 0.02755021]
Beta Covariance: [[ 0.13114351 -0.01018346  0.01662103]
 [-0.01018346  0.03803358 -0.13999876]
 [ 0.01662103 -0.13999876  0.52036139]]
Residual Variance: 0.0014586287991356155
Inverse Condition #: 0.0016487975837974767
Reason(s) for Halting:
  Sum of squares convergence


ODR: logR =  0.765(7) logL - 0.426(14) logM - 0.421(28)
MLR: logR =  0.737(7) logL - 0.371(14) logM - 0.323(27)

In [56]:
#P0Stars, P1Stars = PulsationModeSort(DataToFit)
DataToFit = RawData
LowLogMToFitStars, HighLogMToFitStars = MassSort(DataToFit)
logMData = DataToFit[logMIndex]
logMLT005Data = DataToFit[logMIndex][LowLogMToFitStars]
logMMGT005Data = DataToFit[logMIndex][HighLogMToFitStars]
logRData = DataToFit[logRIndex]
logRMLT005Data = DataToFit[logRIndex][LowLogMToFitStars]
logRMGT005Data = DataToFit[logRIndex][HighLogMToFitStars]
logLData = DataToFit[logLIndex]
logLLT005Data = DataToFit[logLIndex][LowLogMToFitStars]
logLMGT005Data = DataToFit[logLIndex][HighLogMToFitStars]
logP0Data = DataToFit[logP0Index]
logP1Data = DataToFit[logP1Index]

In [58]:
# Create a RealData object using our initiated data from above.
data = RealData(np.asarray([logMData,logLData]), logP0Data) # Data: x=[logM,logL], y=logP

# Set up ODR with the model and data.
odr = ODR(data, linear_model, beta0=[0., 0.,0.])

# Run the regression.
out = odr.run()

# Use the in-built pprint method to give us results.
out.pprint()

Beta: [-1.69612259  1.62323946 -3.42559931]
Beta Std Error: [0.01789202 0.01026696 0.03636982]
Beta Covariance: [[ 0.20249182 -0.0690122   0.22224032]
 [-0.0690122   0.06667646 -0.23528771]
 [ 0.22224032 -0.23528771  0.83670213]]
Residual Variance: 0.0015809257303748207
Inverse Condition #: 0.012415399030554013
Reason(s) for Halting:
  Sum of squares convergence


In [59]:
# Create a RealData object using our initiated data from above.
data = RealData(np.asarray([logMData,logLData]), logRData) # Data: x=[logM,logL], y=logR

# Set up ODR with the model and data.
odr = ODR(data, linear_model, beta0=[0., 1.,2.])

# Run the regression.
out = odr.run()

# Use the in-built pprint method to give us results.
out.pprint()

Beta: [-0.39553899  0.78086677 -0.51795742]
Beta Std Error: [0.00898    0.00521249 0.01850179]
Beta Covariance: [[ 0.05194133 -0.01737576  0.05579953]
 [-0.01737576  0.01750055 -0.0618758 ]
 [ 0.05579953 -0.0618758   0.22048961]]
Residual Variance: 0.0015525275853659048
Inverse Condition #: 0.0017144027544574922
Reason(s) for Halting:
  Sum of squares convergence
